In [1]:
#1.Import the library of Python
import pandas as pd
import numpy as np

In [2]:
housing = pd.read_csv("housing.csv")

In [3]:
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
housing.tail()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND
20639,-121.24,39.37,16.0,2785.0,616.0,1387.0,530.0,2.3886,89400.0,INLAND


In [5]:
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [6]:
# 2. Importing Scikit-Learn And its Tools

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [7]:
# 3. Creating the Stratified test set Based on Income Category

housing["income_cat"] = pd.cut(housing['median_income'],bins=[0,1.5,3.0,4.5,6.0,np.inf],labels=[1,2,3,4,5])
Split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index,test_index in Split.split(housing,housing["income_cat"]):
    Strat_train_set = housing.loc[train_index].drop("income_cat",axis=1)
    Strat_test_set = housing.loc[test_index].drop("income_cat",axis=1)
    

In [8]:
# 4. Work on the copy of Training Data

housing = Strat_train_set.copy()

In [9]:
# 4.Seprate the Predictor and Labels

housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value",axis=1)

In [10]:
# 4.Seprate Numerical And Categorial Columns

num_attribs = housing.drop("ocean_proximity",axis=1).columns.tolist()
cat_attribs = ["ocean_proximity"]

In [11]:
# 5.Creating the Pipelines

# Numerical Pipeline
num_pipeline = Pipeline([
               ("impute",SimpleImputer(strategy="median")),
               ("scalar",StandardScaler())
            ])

# Categorical Pipeline
cat_pipeline = Pipeline([
               ("onehot",OneHotEncoder(handle_unknown="ignore"))
         ])


# Full Pipeline
full_pipeline = ColumnTransformer([
                ("num",num_pipeline,num_attribs),
                ("cat",cat_pipeline,cat_attribs)
])

In [12]:
# 6. Transform the Data 

housing_prepared = full_pipeline.fit_transform(housing)

In [13]:
# 7.Display the Numpy array

housing_prepared

array([[-0.94135046,  1.34743822,  0.02756357, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.17178212, -1.19243966, -1.72201763, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.26758118, -0.1259716 ,  1.22045984, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-1.5707942 ,  1.31001828,  1.53856552, ...,  0.        ,
         0.        ,  0.        ],
       [-1.56080303,  1.2492109 , -1.1653327 , ...,  0.        ,
         0.        ,  0.        ],
       [-1.28105026,  2.02567448, -0.13148926, ...,  0.        ,
         0.        ,  0.        ]])

In [14]:
# 8. Train the Model

#linear Regression Model
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared,housing_labels)
lin_preds = lin_reg.predict(housing_prepared)
lin_rsme = root_mean_squared_error(housing_labels,lin_preds)
# print(f"The root mean squared error for linear regression is {lin_rsme}")
lin_rmses = -cross_val_score(lin_reg,housing_prepared,housing_labels,scoring='neg_root_mean_squared_error',cv=10)
print (pd.Series(lin_rmses).describe())


# Decision Tree Model
dec_reg = DecisionTreeRegressor()
dec_reg.fit(housing_prepared,housing_labels)
dec_preds = dec_reg.predict(housing_prepared)
# dec_rsme = root_mean_squared_error(housing_labels,dec_preds)
dec_rmses = -cross_val_score(dec_reg,housing_prepared,housing_labels,scoring='neg_root_mean_squared_error',cv=10)
# print(f"The root mean squared error for Decision Tree {dec_rsme}")
print (pd.Series(dec_rmses).describe())

# Random Forest Model
random_forest_reg = RandomForestRegressor()
random_forest_reg.fit(housing_prepared,housing_labels)
random_forest_preds = random_forest_reg.predict(housing_prepared)
random_forest_rsme = root_mean_squared_error(housing_labels,random_forest_preds)
# print(f"The root mean squared error for Random Forest is {random_forest_rsme}")
random_forest_rmses = -cross_val_score(random_forest_reg,housing_prepared,housing_labels,scoring='neg_root_mean_squared_error',cv=10)
print (pd.Series(random_forest_rmses).describe())


count       10.000000
mean     69204.322755
std       2500.382157
min      65318.224029
25%      67124.346106
50%      69404.658178
75%      70697.800632
max      73003.752739
dtype: float64
count       10.000000
mean     69121.169133
std       2319.369468
min      66169.902249
25%      67053.472078
50%      69368.972968
75%      70108.956142
max      73367.611551
dtype: float64
count       10.000000
mean     49386.441854
std       2125.428952
min      45798.876983
25%      47958.369283
50%      49168.934982
75%      50518.693625
max      53174.771450
dtype: float64
